# 04 · Does the text add anything?

Financial ratios already predict credit distress. The question here is not
whether risk-factor text predicts anything — a model with more columns in it
will always fit better. It is whether text carries information the balance
sheet does not **already** contain, measured out of sample.

### The label

Interest coverage = EBIT / interest expense. Below 1, a firm cannot service
its interest out of operating earnings. `distress` is coverage < 1 in year
*t+1*, predicted from year-*t* information.

The sample is restricted to firms with coverage >= 1 at *t*, so the model
predicts **transitions into** distress rather than persistence of a state it
can already see. Leaving already-distressed firms in would inflate every AUC
in this notebook without anyone learning anything.

### The test

Logistic regression on ratios alone, then ratios plus text, on an
out-of-time split. The statistic is the AUC difference.

**A raw improvement proves nothing.** Adding any column moves the AUC;
adding five columns of anything moves it more. So the delta is judged
against a **permutation null**: the text block is shuffled across firms
within each year, which destroys the firm-specific link while leaving every
marginal distribution, the year mix, the model form and the parameter count
untouched. Refit, recompute, repeat. That distribution is what this delta
looks like when the text carries nothing.

### The controls come first

A decision rule that cannot reject noise certifies nothing, and one that
cannot detect a known driver has no power. Both controls run **before** the
real verdict, and if either misbehaves everything below it is void.

In [ ]:
import warnings, importlib
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import config, viz, signal_test
importlib.reload(signal_test)
from signal_test import (build_panel, attrition, delta_auc, permutation_test,
                         noise_control, report, report_noise,
                         RATIO_COLS, TEXT_COLS)
viz.use_style()
print(config.describe())

## Assemble the panel

In [ ]:
fund = pd.read_parquet(config.INTERIM / "fundamentals.parquet")
text = pd.read_parquet(config.PROCESSED / "text_features.parquet")
print(f"fundamentals : {len(fund):,} firm-years, {fund.cik.nunique():,} firms")
print(f"text features: {len(text):,} firm-years")

panel = build_panel(fund, text)
print()
print(f"modelling panel: {len(panel):,} firm-years, "
      f"{panel.cik.nunique():,} firms")
print(f"distress rate  : {panel.distress.mean():.2%}")
print()
print(panel.groupby("filing_year")["distress"]
      .agg(["size","mean"]).rename(columns={"size":"n","mean":"distress_rate"})
      .round(4).to_string())

### Where did the rows go?

A panel that arrives far smaller than its inputs is the commonest way an
analysis quietly loses its power, and a single row count will not say which
filter did it.

This table is here because the first version of this notebook produced a
panel of **408 firm-years from 197 firms** out of 8,791 — a 48-row test set
with about five distress events. The positive control correctly refused to
certify it: no test has power at that size. The cause was the label. It used
interest coverage, which needs `InterestExpense` tagged at both *t* and
*t+1*, and most filers report interest only in the notes.

In [ ]:
display(attrition(fund, text))

**Read the distress rate before anything else.** Below about 2% the
positive class is too thin for a stable out-of-time AUC and the whole test
becomes noise; above about 40% the coverage filter is not doing its job. If
the per-year counts collapse in the final year, the label's forward shift
has eaten it — that year has no *t+1* to look at.

In [ ]:
split = int(panel["filing_year"].max())
tr = panel["filing_year"] < split
print(f"train : {panel.loc[tr,'filing_year'].min():.0f}-"
      f"{panel.loc[tr,'filing_year'].max():.0f}   "
      f"{tr.sum():,} rows, {panel.loc[tr,'distress'].mean():.2%} distress")
print(f"test  : {split}              "
      f"{(~tr).sum():,} rows, {panel.loc[~tr,'distress'].mean():.2%} distress")
print()
print("out-of-time, not random: two firm-years from the same year share")
print("macro conditions, so a random holdout leaks.")

### Is any feature crushed?

Univariate out-of-time AUC for each feature on its own. This is a cheap
check with a specific target: a feature whose AUC sits at ~0.50, or below
it, is not necessarily uninformative — it may be **crushed by outliers**.

Winsorising Assets and EBIT separately does not bound their quotient. One
firm with near-zero assets yields an ROA in the hundreds, and after
standardisation that single row compresses every other value toward zero.
Tested in isolation, 15 such rows in 1,500 moved ROA's AUC from 0.80 to
**0.20** — the feature does not merely weaken, it inverts. The ratios are
therefore winsorised at 1/99 as well as their inputs.

An AUC below 0.50 here means the feature predicts the *opposite* of the
label, which is worth understanding before it goes anywhere near a
model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

tr_m, te_m = (panel["filing_year"] < split).to_numpy(), (panel["filing_year"] == split).to_numpy()
rows = []
for c in RATIO_COLS + TEXT_COLS:
    x = panel[[c]].to_numpy()
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    m.fit(x[tr_m], panel.loc[tr_m, "distress"])
    a = roc_auc_score(panel.loc[te_m, "distress"], m.predict_proba(x[te_m])[:, 1])
    rows.append({"feature": c, "univariate_auc": round(a, 4),
                 "min": round(panel[c].min(), 4), "max": round(panel[c].max(), 4),
                 "flag": "CRUSHED?" if a < 0.52 else ""})
display(pd.DataFrame(rows).sort_values("univariate_auc", ascending=False))

## Control 1 — pure noise. Must stay CALIBRATED.

Blocks of Gaussian noise the same width as the text block.

### Why this is fifty draws and not one

The first version tested a **single** noise realisation and asked whether it
was promoted. It was: z = +2.24, p = 0.0100, and the gate declared the
entire notebook void.

The test was not broken. Testing one noise draw at alpha = 0.05 promotes it
about **5% of the time by construction** — that is what a 5% false-positive
rate *means*. Read as a pass/fail gate on one draw, the negative control is
a coin flip that voids the analysis once in twenty runs, and re-running
until it passes would be the worst possible response.

The diagnostic evidence was already visible in that failing output: the
null mean was **-0.0076**, correctly showing that five junk columns normally
*cost* accuracy. The draw simply landed in the upper tail.

So the question changes from *"was this draw promoted?"* to **"is noise
promoted at roughly the nominal rate?"**

### And judged by a binomial test, not by eye

The first attempt at this tolerated "up to 3x nominal" — and duly passed a
run showing **3 of 20 (15%) promoted**, purely because `0.15 <= 0.15`. An
arbitrary threshold dressed up as a criterion is not a criterion.

The honest question is whether the observed count would be surprising *if
the true false-positive rate really were alpha*, which is a one-sided
binomial test. That also exposes the sample-size problem: at 20 draws the
standard error on a 5% rate is about 4.9%, so 5% and 15% are barely
distinguishable. Hence 50 draws.

In [ ]:
r_noise = noise_control(panel, RATIO_COLS, width=len(TEXT_COLS),
                        test_year=split, n_draws=50, n_perm=100,
                        seed=config.RANDOM_SEED)
report_noise(r_noise)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.hist(r_noise["p_values"], bins=20, range=(0, 1), color=viz.SERIES[0],
        edgecolor=viz.SURFACE, linewidth=0.5)
ax.axvline(r_noise["alpha"], color=viz.SERIES[7], linewidth=2)
ax.annotate(f"  alpha={r_noise['alpha']}", (r_noise["alpha"], ax.get_ylim()[1]*0.85),
            color=viz.SERIES[7], fontsize=9, fontweight="600")
ax.set_title("Negative control: p-values across independent noise draws")
ax.set_xlabel("p-value"); ax.set_ylabel("draws")
plt.show()
print("under a calibrated test these are roughly uniform on [0,1]")

## Control 2 — current profitability. Must PASS.

`roa` (EBIT / assets) is held out of the baseline and offered back as a
candidate. A firm earning 0.1% on assets is far closer to an operating loss
than one earning 20%, so if the test cannot detect it, the test has no
power.

### Why this is not leverage

The first version used **leverage** here, on the reasoning that leverage
drives credit distress. It failed: delta +0.0005, z +1.36, NO SIGNAL.

That failure was correct, and it was the control catching a mismatch
between itself and the label rather than a broken test. Leverage predicts
**default** and **coverage breaches** — it drives interest expense, which
drives coverage. It says little about whether a firm's *operations* swing
to a loss. When the label changed from a coverage breach to an
operating-loss transition, the control needed to change with it.

A positive control has to be a variable known to drive **the label you
actually chose**, not credit risk in general. Leverage is kept below as a
secondary diagnostic, because "leverage does not predict operating losses"
is a result worth recording rather than discarding.

In [ ]:
base_no_roa = [c for c in RATIO_COLS if c != "roa"]
roa = panel[["roa"]].to_numpy()

r_roa = permutation_test(panel, base_no_roa, ["roa"], split,
                         n_perm=200, block=roa)
report("POSITIVE CONTROL — current profitability (roa)", r_roa)

#### Leverage, as a secondary diagnostic

In [ ]:
base_no_lev = [c for c in RATIO_COLS if c != "leverage"]
lev = panel[["leverage"]].to_numpy()

r_lev = permutation_test(panel, base_no_lev, ["leverage"], split,
                         n_perm=200, block=lev)
report("leverage (secondary — not the gate)", r_lev)

In [ ]:
controls_ok = (r_noise["calibrated"] and r_roa["verdict"] == "SIGNAL")
print("CONTROLS BEHAVE — verdicts below are meaningful" if controls_ok
      else "CONTROLS FAILED — every verdict below is VOID")

## The actual question

Text block: `risk_change` plus the four Loughran-McDonald category rates.

In [ ]:
r_text = permutation_test(panel, RATIO_COLS, TEXT_COLS, split, n_perm=200)
report("TEXT BLOCK — risk_change + LM rates", r_text)

### Interest coverage, as a diagnostic

Coverage is the better credit metric and is *not* in the feature set,
because requiring it destroys the panel. Reporting how much of the panel it
would have covered makes that trade-off auditable rather than hidden.

In [ ]:
cov = fund.rename(columns={"year":"filing_year"}).merge(
    panel[["cik","filing_year"]], on=["cik","filing_year"], how="inner")
has_int = cov["interest"].notna() & (cov["interest"].abs() > 0)
print(f"panel firm-years with a tagged InterestExpense: "
      f"{has_int.sum():,} of {len(cov):,}  ({has_int.mean():.1%})")
print("that share is why the label is an operating-loss transition")

### And `risk_change` on its own

The headline feature, isolated. If the block passes but this does not, the
signal is in the dictionary rates rather than in the rewriting.

In [ ]:
r_rc = permutation_test(panel, RATIO_COLS, ["risk_change"], split, n_perm=200)
report("risk_change alone", r_rc)

## The null distribution, drawn

The observed delta against the distribution of deltas produced by text that
has been decoupled from the firm it belongs to.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(r_text["null"], bins=40, color=viz.SERIES[0],
        edgecolor=viz.SURFACE, linewidth=0.5, label="permutation null")
ax.axvline(r_text["delta"], color=viz.SERIES[1], linewidth=2)
ax.annotate(f"  observed {r_text['delta']:+.4f}",
            (r_text["delta"], ax.get_ylim()[1]*0.88),
            color=viz.SERIES[1], fontsize=9, fontweight="600")
ax.set_title("Text block: observed AUC gain vs permutation null")
ax.set_xlabel("delta AUC (ratios + text) - (ratios)")
ax.set_ylabel("permutations")
plt.show()

## Verdict

Whatever this says, it is the result. A null finding here is not a failed
project — "text adds nothing once you control for the balance sheet, and
here is the null distribution that shows it" is a stronger claim than an
unguarded AUC improvement, because it is one almost nobody bothers to
test.

In [ ]:
summary = pd.DataFrame([
    {"block": f"noise x{r_noise['n_draws']} (neg control)",
     "auc_base": np.nan, "auc_augmented": np.nan,
     "delta": r_noise["mean_delta"], "z": np.nan,
     "p_value": r_noise["promotion_rate"], "verdict": r_noise["verdict"]},
    {"block": "roa (pos control)", **{k: r_roa[k] for k in
        ("auc_base","auc_augmented","delta","z","p_value","verdict")}},
    {"block": "leverage (diagnostic)", **{k: r_lev[k] for k in
        ("auc_base","auc_augmented","delta","z","p_value","verdict")}},
    {"block": "text block", **{k: r_text[k] for k in
        ("auc_base","auc_augmented","delta","z","p_value","verdict")}},
    {"block": "risk_change alone", **{k: r_rc[k] for k in
        ("auc_base","auc_augmented","delta","z","p_value","verdict")}},
]).round(4)
display(summary)

summary.to_csv(config.PROCESSED / "signal_test_results.csv", index=False)
print(f"saved: {config.PROCESSED / 'signal_test_results.csv'}")